In [19]:
!pip install opencv-python-headless
!pip install opencv-python
!pip install scipy
!pip3 install pandas
!pip install matplotlib seaborn


[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: pip install --upgrade pip
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 1.4 MB/s eta 0:00:001.4 MB/s eta 0:00:010m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 1.7 MB/s eta 0:00:00 MB/s eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 1.7 MB/s eta 0:00:00 MB/s eta 0:00:01:01

[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: pip install --upgrade pip


In [4]:
import scipy.io
import cv2
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import random 
import numpy as np
import json
import math
import traceback

In [5]:
subject_colors = {}

In [6]:
def parseVideoName(video_name):
    video_name = video_name.replace('.avi', '')
    first_part,second_part = video_name.split(',')
    video_type, num_targets = first_part.rsplit('_', 1)
    videos_info = second_part.split('_')
    
    parsed_data = {
        'video_name': video_name,
        'target_name': second_part,
        'video_type': video_type,
        'num_targets': int(num_targets),
        'first_video': int(videos_info[0]),
        'second_video': int(videos_info[1]),
        'first_start_frame': int(videos_info[2]),
        'second_start_frame': int(videos_info[3]),
        'first_rotation': float(videos_info[4]),
        'second_rotation': float(videos_info[5]),
        'full_rotation': int(videos_info[6] if len(videos_info) > 6 else 0),
    }
    
    return parsed_data
    
def generateRandomColor():
    return (random.randint(0, 255), random.randint(0, 255), random.randint(0, 255))

def processTrialsVideos(file_path):
    df = pd.read_csv(file_path)
    parsed_rows = []
    
    for index, row in df.iterrows():
        video_name = row['videoName']
        parsed_data = parseVideoName(video_name)
        parsed_data['firstTargets'] = row['firstTargets']
        parsed_data['secondTargets'] = row['secondTargets']
        parsed_data['fullRotation'] = row['rotAngle']
        parsed_data['clickedItemsfromFirstVid'] = row['clickedItemsfromFirstVid']
        parsed_data['clickedItemsfromSecondVid'] = row['clickedItemsfromSecondVid']
        parsed_rows.append(parsed_data)

    return pd.DataFrame(parsed_rows)
    
def loadMatFile():
    root_folder = 'GazeMatV24'
    test_subs = [f"sub{num}{i}" for num in range(1, 31) for i in range(1, 5)]

    data = {}
    
    for sub_dir in os.listdir(root_folder):
        if sub_dir in test_subs:
            mat_file_path = os.path.join(root_folder, sub_dir, 'finalCleanSampled.mat')
            if os.path.exists(mat_file_path):
                mat_data = scipy.io.loadmat(mat_file_path)
                data[sub_dir] = mat_data['itemsSampled']
            else:
                print(f"{mat_file_path} does not exist.")
    print("Data loaded successfully.")
    return data


def loadShapeFishPoints(videoName,numberFrame):
    # before was + 1 , i removed it
    json_filename = f'img{numberFrame:03d}.json'

    json_path = os.path.join('./jsons', str(videoName), json_filename)
    # print(f"Checking path: {json_path}")
    if not os.path.exists(json_path):
        raise FileNotFoundError(f"JSON file not found: {json_path}")

    with open(json_path, 'r') as json_file:
        data = json.load(json_file)
    
    return data.get('shapes', [])

def getTargetsFromJson(shapes,targets):
    if not isinstance(targets, (list, tuple)):
        targets = [targets]
    
    target_digits = set()
    for target in targets:
        if isinstance(target, int):
            target_digits.update(str(target))
        else:
            raise ValueError("Targets should be integers or lists/tuples of integers")

    filtered_shapes = [shape for shape in shapes if shape['label'] in target_digits]
    
    return filtered_shapes

def generateSubjectName(subIndex):
    number_part = subIndex[3:]
    modified_number = number_part[:-1] if len(number_part) > 1 else '0'
    return f'sub{modified_number}'

def getVideoInfo(video):
    fps = video.get(cv2.CAP_PROP_FPS)
    width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    return fps, width, height 

def createOutputVideo(video,name):
    output_gaze_videos = 'gazeOutputVideoNew'
    os.makedirs(output_gaze_videos, exist_ok=True)
    fps, width, height = getVideoInfo(video)
    
    output_gaze_videos_path = os.path.join(output_gaze_videos, name + '.avi')
    output_video = cv2.VideoWriter(output_gaze_videos_path, cv2.VideoWriter_fourcc(*'XVID'), fps, (width, height))
    return output_video

def getSubjectColor(subject):
    if subject not in subject_colors:
        subject_colors[subject] = generateRandomColor()
    return subject_colors[subject]

def openVideo(video):
    trialVideo = cv2.VideoCapture(os.path.join('./videos', video.video_type, video.target_name + '.avi'))
    
    if not trialVideo.isOpened():
        raise ValueError(f"Unable to open video file: {os.path.join('./videos', video.video_type, video.target_name + '.avi')}")
    return trialVideo

def generateXPoints(x):
    return int(x) - 630

def generateYPoints(y):
    return int(y) - 210


def getTotalShapeFirst(shapeInfo,firstFrame):
    firstTargets = shapeInfo['firstTargets']
    shapes_video1 = loadShapeFishPoints(shapeInfo['first'], firstFrame)
    return getTargetsFromJson(shapes_video1,firstTargets)
    
def getTotalShapeSecond(shapeInfo,secondFrame):
    secondTargets = shapeInfo['secondTargets']
    shapes_video2 = loadShapeFishPoints(shapeInfo['second'], secondFrame)
    merged_shapes = getTargetsFromJson(shapes_video2,secondTargets)
    return merged_shapes

def rotate_frame(frame, angle, center):
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    rotated_frame = cv2.warpAffine(frame, M, (frame.shape[1], frame.shape[0]))
    return rotated_frame

def adjust_shape_points(shape, x_offset=90, y_offset=10):
    adjusted_points = []
    for point in shape['points']:
        adjusted_point = [point[0] - x_offset, point[1] - y_offset]
        adjusted_points.append(adjusted_point)
    shape['points'] = adjusted_points
    return shape
    
def draw_shapes(frame, shapes, typeV, color=(0, 0, 255)):
    data = {
        1: [90, 5],
        7: [25,25],
        10: [25,15]
    }
    overlay = frame.copy()
    opacity = 0.1
    for shape in shapes:
        shape = adjust_shape_points(shape,data[typeV][0],data[typeV][1])  # Adjust the points
        points = shape['points']

        if not points:
            continue
        
        np_points = np.array(points, dtype=np.int32).reshape((-1, 1, 2))

        if np_points.size == 0:
            continue
        # x, y = np_points[0][0]  # Position for the text
        # cv2.putText(frame, f"_{typeV}", (x + 20, y + 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 225), 1, cv2.LINE_AA)
        cv2.polylines(overlay, [np_points], isClosed=True, color=color, thickness=1)
        cv2.fillPoly(overlay, [np_points], color=color)
    cv2.addWeighted(overlay, opacity, frame, 1 - opacity, 0, frame)

def rotate_frameNew(frame, angle):
    center = (330, 330)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    rotated_frame = cv2.warpAffine(frame, M, (frame.shape[1], frame.shape[0]))
    return rotated_frame

In [7]:
gazeDataList = loadMatFile()

Data loaded successfully.


In [8]:

len(gazeDataList['sub14'][0][1]) # trial id (0 - 39) , (x ,y) 0 ,1 , frame of trial ( 0 - 109)

110

In [9]:
def load_video_info():
    root_folder = './Results'
    subject_folders = [f"sub{num}" for num in range(1, 31)]
    video_info = {}

    for sub_folder in subject_folders:
        for trial_num in range(1, 5):
            
            

            file_path = os.path.join(root_folder, sub_folder, f"{sub_folder}{trial_num}.csv")
            if os.path.exists(file_path):
                
                key = f"{sub_folder}{trial_num}"
                video_info[key] = processTrialsVideos(file_path)
            else:
                print(f"{file_path} does not exist.")
    return video_info

In [10]:
video_info = load_video_info()

In [11]:
video_info['sub11']

,video_name,target_name,video_type,num_targets,first_video,second_video,first_start_frame,second_start_frame,first_rotation,second_rotation,full_rotation,firstTargets,secondTargets,fullRotation,clickedItemsfromFirstVid,clickedItemsfromSecondVid
0,"NO_2,1_10_1_1_45_315",1_10_1_1_45_315,NO,2,1,10,1,1,45.0,315.0,0,12,0,278,21,0
1,"NO_2,1_10_119_344_45_90",1_10_119_344_45_90,NO,2,1,10,119,344,45.0,90.0,0,3,4,117,3,4
2,"OB_4,1_10_1_344_45_315",1_10_1_344_45_315,OB,4,1,10,1,344,45.0,315.0,0,3412,0,283,3124,0
3,"OB_2,10_10_114_229_0_135",10_10_114_229_0_135,OB,2,10,10,114,229,0.0,135.0,0,14,0,170,14,0
4,"NO_4,10_10_229_344_0_315",10_10_229_344_0_315,NO,4,10,10,229,344,0.0,315.0,0,43,41,13,43,14
5,"OB_2,1_10_1_1_45_0",1_10_1_1_45_0,OB,2,1,10,1,1,45.0,0.0,0,0,14,64,0,14
6,"OC_4,1_10_119_344_0_247.5",1_10_119_344_0_247.5,OC,4,1,10,119,344,0.0,247.5,0,13,23,260,13,32
7,"OB_OC_4,1_10_119_114_45_225",1_10_119_114_45_225,OB_OC,4,1,10,119,114,45.0,225.0,0,342,1,171,421,1
8,"OC_4,1_10_119_344_45_90",1_10_119_344_45_90,OC,4,1,10,119,344,45.0,90.0,0,31,41,55,23,14
9,"OB_4,1_10_119_229_45_225",1_10_119_229_45_225,OB,4,1,10,119,229,45.0,225.0,0,342,4,123,324,0


In [12]:
def calculate_accuracy(first_targets, second_targets, clicked_first, clicked_second):

    first_targets_str = str(first_targets)
    clicked_first_str = str(clicked_first)
    second_targets_str = str(second_targets)
    clicked_second_str = str(clicked_second)
    
    def calculate_match_percentage(target_str, clicked_str):
        target_digits = set(target_str)
        clicked_digits = set(clicked_str)
        matches = target_digits.intersection(clicked_digits)
        match_percentage = (len(matches) / len(target_digits)) * 100 if target_digits else 0
        return match_percentage
    
    match_percentage_first = calculate_match_percentage(first_targets_str, clicked_first_str)
    match_percentage_second = calculate_match_percentage(second_targets_str, clicked_second_str)
    
    accuracy = (match_percentage_first + match_percentage_second) / 2
    accuracy = round(accuracy)
    
    return accuracy


In [13]:
def processTrialsVideossss(file_path, video_name):
    df = pd.read_csv(file_path)
    
    video_data = {}
    
    for index, row in df.iterrows():
        subject_name = row['videoName'] 
        
        if subject_name not in video_data:
            video_data[subject_name] = {}

        # print(subject_name)
        remain = parseVideoName(subject_name)

        video_entry = {
            "mainVideoName": subject_name,
            "rotAngle": row['rotAngle'],
            "videoLength": row['videoLength'],
            "firstTargets": row['firstTargets'],
            "secondTargets": row['secondTargets'],
            # parsed_data[''] = row['rotAngle']
            "clickedItemsfromFirstVid": row['clickedItemsfromFirstVid'],
            "clickedItemsfromSecondVid": row['clickedItemsfromSecondVid'],
            "responseTimeFirstVid": [float(x) for x in str(row['respnseTimeFirstVid']).split()],
            "responseTimeSecondVid": [float(x) for x in str(row['respnseTimeSecondVid']).split()],
            "trialIndex": index,
        }
        video_entry.update(remain)


        accuracy = calculate_accuracy(
            video_entry['firstTargets'], 
            video_entry['secondTargets'], 
            video_entry['clickedItemsfromFirstVid'], 
            video_entry['clickedItemsfromSecondVid']
        )
        
        video_entry['accuracy'] = accuracy
        
        video_data[subject_name][video_name] = video_entry

    return video_data



def checkPointValid(point, previous_point, frame_idx, axis):
    if math.isnan(point):
        if previous_point is not None and not math.isnan(previous_point):
            print(f"NaN detected for {axis} at frame {frame_idx}. Using previous {axis} value: {previous_point}")
            return previous_point
        else:
            print(f"NaN detected for {axis} at frame {frame_idx}, but no previous value to replace. Skipping.")
            return 0
    return point
    
def customOpenVideo(videoType,videoName):
    modified_target_name = videoName.rsplit('_', 1)[0] + '.avi'
    trialVideo = cv2.VideoCapture(os.path.join('./videos', videoType, modified_target_name))
    
    if not trialVideo.isOpened():
        raise ValueError(f"Unable to open video file: {os.path.join('./videos',videoType , videoName + '.avi')}")
    return trialVideo
    
def test():
    root_folder = './Results/'
    sub_folders = [f'sub{i}' for i in range(1, 31)]
    translatedCode = {}

    for sub_folder in sub_folders:
        sub_folder_path = os.path.join(root_folder, sub_folder)
    
        csv_files = [f for f in os.listdir(sub_folder_path) if f.endswith('.csv')]
        
        for csv_file in csv_files:
            video_name = csv_file.split('.')[0]  # Extract the video name from the CSV filename
            file_path = os.path.join(sub_folder_path, csv_file)
            
            video_data = processTrialsVideossss(file_path, video_name)
            
            for subject_name, subjects_data in video_data.items():
                
                for video_name, details in subjects_data.items():
                    if video_name == 'sub143':
                        continue;
                    subject_name_rotangle = f"{details['video_name']}_{details['rotAngle']}.avi"
                    if subject_name_rotangle not in translatedCode:
                        translatedCode[subject_name_rotangle] = {}
                    
                    trial_index = details['trialIndex']

                    gaze_data = gazeDataList[video_name][trial_index]
                    details['gazeData'] = gaze_data
                    
                    translatedCode[subject_name_rotangle][video_name] = details

    return translatedCode

In [14]:
aggregated_data = test()

In [15]:
def filter_accurate_data(data):
    filtered_data = {}
    
    for video_name, subjects_data in data.items():
        for subject_name, details in subjects_data.items():
            if details['accuracy'] == 100:
                if video_name not in filtered_data:
                    filtered_data[video_name] = {}
                filtered_data[video_name][subject_name] = details

    return filtered_data


In [16]:
filtered_data = filter_accurate_data(aggregated_data)

In [22]:
for video_name, subjects in filtered_data.items():
    
    video_info = parseVideoName(video_name)
    # print(f'{subjects}')
    trial_video = customOpenVideo(video_info['video_type'], video_info['target_name'])
    output_video_name = f"{video_info['video_type']+ '_' + str(video_info['num_targets']) + ',' + video_info['target_name']}"
    output_video = createOutputVideo(trial_video, output_video_name)
    frame_idx = 0

    first_subject_key, first_subject_details = next(iter(subjects.items()))
    # print(video_info)

    previous_x = None
    previous_y = None

    while True:
        ret, frame = trial_video.read()
        
        if not ret:
            break
        
        frame = rotate_frameNew(frame, video_info['full_rotation'])

        for subject, details in subjects.items():
            if subject == 'sub143':
                print('exception')
                continue
            try:
                x_points = gazeDataList[subject][details['trialIndex']][0]
                y_points = gazeDataList[subject][details['trialIndex']][1]
                
                # x = checkPointValid(x_points[frame_idx][0], previous_x, frame_idx, 'x')
                # y = checkPointValid(y_points[frame_idx][0], previous_y, frame_idx, 'y')
                
                x = x_points[frame_idx][0]
                y = y_points[frame_idx][0]
                
                # if x is None or y is None:
                #     continue
            
                # previous_x = x
                # previous_y = y
            
                new_x = generateXPoints(x)
                new_y = generateYPoints(y)

                # print(new_x,new_y)
                color = getSubjectColor(subject)

                cv2.circle(frame, (new_x, new_y), 5, color, 3)
                cv2.putText(frame, generateSubjectName(subject), (new_x + 5, new_y - 5),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA)
            except Exception as e:
                print(f"An error occurred at subject {subject}, trial {details['trialIndex']}, frameframe_rotated {frame_idx}: {e}")
                traceback.print_exc()
                
        shapeInfo = {
            'first': video_info['first_video'],
            'second': video_info['second_video'],
            'firstTargets': first_subject_details['firstTargets'],
            'secondTargets': first_subject_details['secondTargets'],
            'firstStart': video_info['first_start_frame'],
            'secondStart': video_info['second_start_frame']
        }
            
        shapeFirst = getTotalShapeFirst(shapeInfo, shapeInfo['firstStart'] + frame_idx)
        shapeSecond = getTotalShapeSecond(shapeInfo, shapeInfo['secondStart'] + frame_idx)
        
        blank_frame = np.zeros_like(frame)
        blank_frame2 = np.zeros_like(frame)
        draw_shapes(blank_frame, shapeFirst, video_info['first_video'])
        draw_shapes(blank_frame2, shapeSecond, video_info['second_video'])

        center = (330, 330)
        rotated_shapes_frame = rotate_frame(blank_frame, video_info['first_rotation'], center)

        full_rotated_first = rotate_frameNew(rotated_shapes_frame, video_info['full_rotation'])
        rotated_shapes_frame2 = rotate_frame(blank_frame2, video_info['second_rotation'], center)
        full_rotated_second = rotate_frameNew(rotated_shapes_frame2, video_info['full_rotation'])

        mask = full_rotated_first > 0
        frame[mask] = full_rotated_first[mask]

        mask2 = full_rotated_second > 0
        frame[mask2] = full_rotated_second[mask2]
        
        resized_frame = cv2.resize(frame, (660, 660))
        output_video.write(resized_frame)     
        frame_idx += 1
        
    trial_video.release()    
    output_video.release()
    cv2.destroyAllWindows()

KeyboardInterrupt: 

In [ ]:
# gaze_array = np.zeros((9610, 115))

# for subject in gazeDataList.keys():
#     subject_num = int(subject[3:])  
#     for trialId in range(40):
#         base_index = trialId * 2
#         print('subject_num',subject_num,trialId,'trialId',base_index)
#         for frame_id in range(110):
#             x = gazeDataList[subject][trialId][0][frame_id][0] 
#             y = gazeDataList[subject][trialId][1][frame_id][0] 

#             gaze_array[base_index,frame_id,] = round(x,2) if not np.isnan(x) else 0  # X
#             gaze_array[base_index + 1,frame_id] = round(y,2) if not np.isnan(y) else 0  # Y
            
#             gaze_array[base_index, 113] = subject_num
#             gaze_array[base_index + 1, 113] = subject_num
            
#             gaze_array[base_index, 114] = trialId
#             gaze_array[base_index + 1, 114] = trialId  

In [50]:
def calculate_mean_coordinates(targets):
    means = []
    for target in targets:
        points = target['points']
        if points:
            mean_x = np.mean([p[0] for p in points])
            mean_y = np.mean([p[1] for p in points])
            means.append((round(mean_x,2), round(mean_y,2)))
    return means

In [51]:
gaze_array = np.zeros((9601, 115))

current_index = 0

for video_name, video_details in filtered_data.items():
    for subject, subject_details in video_details.items():
        trial_id = subject_details['trialIndex']
        for frame_id in range(110):
            try:
                if subject == 'sub11':
                    shapeFirst = loadShapeFishPoints(subject_details['first_video'], subject_details['first_start_frame'] + frame_id)
                    shapeSecond = loadShapeFishPoints(subject_details['second_video'], subject_details['second_start_frame'] + frame_id)
                    xx = getTargetsFromJson(shapeFirst,subject_details['firstTargets'])
                    yy = getTargetsFromJson(shapeSecond,subject_details['secondTargets'])
                    print(subject_details['second_video'], subject_details['second_start_frame'] + frame_id)
                    print(xx,yy,'frame_id',frame_id,'trial_id', trial_id)
                    print(calculate_mean_coordinates(xx),calculate_mean_coordinates(yy))
                    meanXXTargets = calculate_mean_coordinates(xx)
                    meanYYTargets = calculate_mean_coordinates(yy)
                    
                x = subject_details['gazeData'][0][frame_id][0]
                y = subject_details['gazeData'][1][frame_id][0]

                gaze_array[current_index, frame_id] = round(x, 2)
                gaze_array[current_index + 1, frame_id] = round(y, 2)

                subject_num = int(subject[3:])
                
                gaze_array[current_index, 111] = 0  # is X 
                gaze_array[current_index + 1, 111] = 1  # is Y
                
                gaze_array[current_index, 113] = subject_num  # subject number
                gaze_array[current_index + 1, 113] = subject_num  # subject number

                gaze_array[current_index, 114] = trial_id  # trial number  
                gaze_array[current_index + 1, 114] = trial_id   # trial number
            except IndexError:
                traceback.print_exc()
                print(f"Frame {frame_id} out of range for trial {trial_id} of subject {subject}")

        # Update the current_index for the next subject
        current_index += 2
print(gaze_array[0])


10 1
[{'label': '1', 'line_color': None, 'fill_color': None, 'points': [[305, 56], [323, 73], [219, 139]]}, {'label': '2', 'line_color': None, 'fill_color': None, 'points': [[684, 227], [705, 228], [702, 280], [722, 330], [691, 286]]}] [] frame_id 0 trial_id 0
[(np.float64(282.33), np.float64(89.33)), (np.float64(700.8), np.float64(270.2))] []
10 2
[{'label': '1', 'line_color': None, 'fill_color': None, 'points': [[305, 56], [323, 73], [219, 139]]}, {'label': '2', 'line_color': None, 'fill_color': None, 'points': [[673, 200], [690, 197], [701, 255], [707, 298], [693, 263]]}] [] frame_id 1 trial_id 0
[(np.float64(282.33), np.float64(89.33)), (np.float64(692.8), np.float64(242.6))] []
10 3
[{'label': '2', 'line_color': None, 'fill_color': None, 'points': [[666, 193], [685, 187], [696, 245], [708, 295], [690, 252]]}, {'label': '1', 'line_color': None, 'fill_color': None, 'points': [[341, 54], [345, 73], [294, 77], [256, 116], [285, 71]]}] [] frame_id 2 trial_id 0
[(np.float64(689.0), np.f

In [34]:
gaze_array.shape

(9601, 115)

In [35]:
from scipy.io import savemat

In [36]:
np.savetxt('gaze_dataNew.csv', gaze_array, delimiter=',', fmt='%0.2f')

In [48]:
ttt = [{'label': '3', 'line_color': None, 'fill_color': None, 'points': [[728, 342], [707, 339], [708, 374], [703, 396], [700, 430], [697, 455], [704, 435], [712, 401], [717, 370]]}]

In [49]:
calculate_mean_coordinates(ttt)[0]

(np.float64(708.44), np.float64(393.56))